# 🎬 YouTube AI Summarizer — Kaggle Backend

This notebook is the **backend**: it loads `facebook/bart-large-cnn` on Kaggle's free GPU,
wraps it in a FastAPI server, and exposes it publicly through ngrok so your local
Streamlit app can call it.

**Before running:**
1. Enable GPU: `Settings → Accelerator → GPU T4 x2` (or any GPU option).
2. Add your secrets: `Add-ons → Secrets`, then add:
   - `NGROK_TOKEN` — free token from https://dashboard.ngrok.com/get-started/your-authtoken
   - `API_KEY` — any password you make up, used to protect your `/summarize` endpoint
   - (optional but recommended) `WEBSHARE_USERNAME` and `WEBSHARE_PASSWORD` — see the proxy note below
3. Run all cells top to bottom, then **keep this notebook session open** — the API only
   stays alive while the session is running.

### ⚠️ Why you need a proxy even on Kaggle
YouTube blocks requests coming from cloud-provider IPs (AWS, GCP, Azure — and Kaggle
notebooks run on Google Cloud too). Without a proxy you'll likely hit
`IpBlocked` / `RequestBlocked` errors, especially after a few requests. A cheap
residential proxy (e.g. [Webshare](https://www.webshare.io/), a couple dollars/month)
fixes this reliably. The notebook below works with or without one — if you don't add
Webshare secrets it just tries a direct connection.


## 1. Install dependencies

In [31]:
!pip install -q \
transformers==4.52.4 \
accelerate \
sentencepiece \
youtube-transcript-api \
fastapi \
uvicorn \
pyngrok \
nltk

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## 2. Imports

In [32]:
import re
import time
import socket
import threading
from urllib.parse import urlparse, parse_qs

import torch
import nltk
from nltk.tokenize import sent_tokenize

from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import (
    TranscriptsDisabled,
    NoTranscriptFound,
    VideoUnavailable,
    IpBlocked,
    RequestBlocked,
)
from youtube_transcript_api.proxies import WebshareProxyConfig

from transformers import pipeline, AutoTokenizer

from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel

from pyngrok import ngrok, conf
import uvicorn

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## 3. Load secrets (never hardcode these!)

This reads your tokens from Kaggle's encrypted Secrets store instead of writing them
in plain text — so you can safely share or push this notebook without leaking anything.

In [33]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

NGROK_TOKEN = secrets.get_secret("NGROK_TOKEN")
API_KEY = secrets.get_secret("API_KEY")

# Optional — leave these secrets unset in Kaggle if you don't want a proxy yet
try:
    WEBSHARE_USERNAME = secrets.get_secret("WEBSHARE_USERNAME")
    WEBSHARE_PASSWORD = secrets.get_secret("WEBSHARE_PASSWORD")
except Exception:
    WEBSHARE_USERNAME, WEBSHARE_PASSWORD = None, None

PROXY_CONFIG = None
if WEBSHARE_USERNAME and WEBSHARE_PASSWORD:
    PROXY_CONFIG = WebshareProxyConfig(
        proxy_username=WEBSHARE_USERNAME, proxy_password=WEBSHARE_PASSWORD
    )
    print("Proxy configured ✅")
else:
    print("No proxy configured — direct connection (may get IP-blocked by YouTube).")

No proxy configured — direct connection (may get IP-blocked by YouTube).


## 4. Load the summarization model

In [34]:
MODEL_NAME = "facebook/bart-large-cnn"

device = 0 if torch.cuda.is_available() else -1
print("Using GPU" if device == 0 else "Using CPU (enable GPU in Settings for speed)")

summarizer = pipeline("summarization", model=MODEL_NAME, device=device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Model loaded successfully")

Using GPU


Device set to use cuda:0


Model loaded successfully


## 5. Transcript extraction (with proxy + friendly errors)

In [35]:
def extract_video_id(url: str) -> str:
    parsed = urlparse(url)
    if "youtu.be" in parsed.netloc:
        return parsed.path.lstrip("/")
    qs = parse_qs(parsed.query)
    ids = qs.get("v")
    if not ids:
        raise ValueError("Invalid YouTube URL")
    return ids[0]


def get_transcript(url: str) -> str:
    video_id = extract_video_id(url)
    api = YouTubeTranscriptApi(proxy_config=PROXY_CONFIG) if PROXY_CONFIG else YouTubeTranscriptApi()
    try:
        transcript = api.fetch(video_id, languages=["en"])
    except (IpBlocked, RequestBlocked):
        raise ValueError(
            "YouTube blocked this request (cloud IP). Add WEBSHARE_USERNAME / "
            "WEBSHARE_PASSWORD secrets to this notebook and re-run."
        )
    except (TranscriptsDisabled, NoTranscriptFound):
        raise ValueError(
            "This video doesn't have an English transcript available. "
            "Please try a different video."
        )
    except VideoUnavailable:
        raise ValueError("This video is unavailable (private, deleted, or region-locked).")
    text = "\n".join(snippet.text for snippet in transcript)
    return text

## 6. Clean up messy auto-generated captions

In [36]:
def clean_transcript(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\b(\w+)( \1\b)+", r"\1", text, flags=re.IGNORECASE)
    return text.strip()

## 7. Sentence-aware, token-aware chunking

Splits on real sentence boundaries (via NLTK) and packs them up to `max_tokens`
so a sentence is never cut in half.

In [37]:
def chunk_text(text: str, max_tokens: int = 900) -> list:
    sentences = sent_tokenize(text)
    chunks, current_chunk, current_length = [], [], 0

    for sentence in sentences:
        sentence_length = len(tokenizer.encode(sentence, add_special_tokens=False))
        if current_length + sentence_length > max_tokens and current_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk, current_length = [], 0
        current_chunk.append(sentence)
        current_length += sentence_length

    if current_chunk:
        chunks.append(" ".join(current_chunk))
    return chunks

## 8. Summarize each chunk (dynamic length + beam search)

In [38]:
def summarize_chunks(chunks: list) -> list:
    summaries = []
    for chunk in chunks:
        input_length = len(tokenizer.encode(chunk, add_special_tokens=False))
        dynamic_max = min(220, max(60, int(input_length * 0.6)))
        dynamic_min = min(100, max(30, int(dynamic_max * 0.5)))

        result = summarizer(
            chunk,
            max_length=dynamic_max,
            min_length=dynamic_min,
            length_penalty=1.0,
            num_beams=4,
            early_stopping=True,
            do_sample=False,
        )
        summaries.append(result[0]["summary_text"])
    return summaries

## 9. Deduplicate + hierarchical final summary

If the combined chunk summaries are still long, summarize them **again** so the
final result reads as one coherent summary instead of stitched-together pieces.

In [39]:
def remove_duplicate_sentences(text: str) -> str:
    seen, cleaned = set(), []
    for sentence in text.split("."):
        sentence = sentence.strip()
        if sentence and sentence not in seen:
            cleaned.append(sentence)
            seen.add(sentence)
    return ". ".join(cleaned)


def final_summary(summaries: list) -> str:
    combined = remove_duplicate_sentences(" ".join(summaries))
    combined_length = len(tokenizer.encode(combined, add_special_tokens=False))

    if combined_length < 700:
        return combined

    combined_chunks = chunk_text(combined, max_tokens=900)
    final_parts = []
    for chunk in combined_chunks:
        input_length = len(tokenizer.encode(chunk, add_special_tokens=False))
        dynamic_max = min(350, max(100, int(input_length * 0.6)))
        dynamic_min = min(180, max(50, int(dynamic_max * 0.5)))
        result = summarizer(
            chunk,
            max_length=dynamic_max,
            min_length=dynamic_min,
            length_penalty=1.0,
            num_beams=4,
            early_stopping=True,
            do_sample=False,
        )
        final_parts.append(result[0]["summary_text"])
    return " ".join(final_parts)

## 10. Orchestrator

In [40]:
MAX_CHUNKS = 15  # raise this if you have a longer GPU session budget

def summarize_youtube(url: str, transcript: str = None) -> str:
    # If the frontend already fetched the transcript (via /transcript), reuse it
    # instead of hitting YouTube again — saves a request and avoids double IP-block risk.
    if not transcript:
        transcript = get_transcript(url)
    transcript = clean_transcript(transcript)

    if not transcript.strip():
        raise ValueError("No speech content found in this video.")

    chunks = chunk_text(transcript)
    if len(chunks) > MAX_CHUNKS:
        raise ValueError(
            f"This video is too long to summarize (needs {len(chunks)} chunks, "
            f"max supported is {MAX_CHUNKS}). Try a shorter video."
        )

    summaries = summarize_chunks(chunks)
    return final_summary(summaries)

## 11. FastAPI app

Two endpoints now:
- `POST /transcript` — just fetches + cleans the transcript, no model involved, fast.
- `POST /summarize` — summarizes. Accepts an optional `transcript` field so it can
  reuse one already fetched via `/transcript` instead of calling YouTube again.

In [41]:
from typing import Optional

class TranscriptRequest(BaseModel):
    youtube_url: str

class SummarizeRequest(BaseModel):
    youtube_url: str
    transcript: Optional[str] = None


In [42]:
app = FastAPI(title="YouTube AI Summarizer", version="1.0")


@app.get("/")
def home():
    return {"status": "running", "model": MODEL_NAME}


@app.post("/transcript")
async def transcript_endpoint(request: TranscriptRequest, authorization: str = Header(None)):
    if authorization != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")
    try:
        raw = get_transcript(request.youtube_url)
        cleaned = clean_transcript(raw)
        if not cleaned.strip():
            raise ValueError("No speech content found in this video.")
        return JSONResponse({
            "status": "success",
            "transcript": cleaned,
            "word_count": len(cleaned.split()),
        })
    except ValueError as e:
        return JSONResponse(status_code=400, content={"status": "error", "message": str(e)})
    except Exception as e:
        return JSONResponse(status_code=500, content={"status": "error", "message": str(e)})


@app.post("/summarize")
async def summarize(request: SummarizeRequest, authorization: str = Header(None)):
    if authorization != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")
    try:
        summary = summarize_youtube(request.youtube_url, transcript=request.transcript)
        return JSONResponse({"status": "success", "summary": summary})
    except ValueError as e:
        return JSONResponse(status_code=400, content={"status": "error", "message": str(e)})
    except Exception as e:
        return JSONResponse(status_code=500, content={"status": "error", "message": str(e)})

## 12. Start the server + ngrok tunnel

Run this cell last. It prints a public HTTPS URL — copy it into your local
`app.py`'s `PUBLIC_URL` constant.

In [43]:
def free_port():
    s = socket.socket()
    s.bind(("", 0))
    port = s.getsockname()[1]
    s.close()
    return port


conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()  # close any stale tunnels from a previous run

port = free_port()
public_url = ngrok.connect(port).public_url
print("=" * 60)
print("PUBLIC URL (copy this into your local app.py):")
print(public_url)
print("API_KEY is set from your Kaggle secret — use the same value locally.")
print("=" * 60)


def run():
    uvicorn.run(app, host="0.0.0.0", port=port)


threading.Thread(target=run, daemon=True).start()
time.sleep(2)
print("Server is up. Keep this notebook session running to keep the API alive.")

PUBLIC URL (copy this into your local app.py):
https://skipper-dissuade-glue.ngrok-free.dev
API_KEY is set from your Kaggle secret — use the same value locally.


INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:55159 (Press CTRL+C to quit)


Server is up. Keep this notebook session running to keep the API alive.


## 13. Quick self-test (optional)

Run this from inside the notebook to confirm everything works before switching
to your local Streamlit app.

In [44]:
import requests

headers = {"Authorization": f"Bearer {API_KEY}"}
test_video = {"youtube_url": "https://www.youtube.com/watch?v=jNQXAC9IVRw"}  # replace with any video

# 1) fetch just the transcript (fast, no GPU used)
t_resp = requests.post(f"{public_url}/transcript", headers=headers, json=test_video, timeout=60)
print("transcript endpoint:", t_resp.status_code)
print(t_resp.json())

# 2) summarize (can reuse the transcript we just fetched to skip a second YouTube request)
transcript_text = t_resp.json().get("transcript")
s_resp = requests.post(
    f"{public_url}/summarize",
    headers=headers,
    json={"youtube_url": test_video["youtube_url"], "transcript": transcript_text},
    timeout=180,
)
print("summarize endpoint:", s_resp.status_code)
print(s_resp.json())

INFO:     34.48.3.188:0 - "POST /transcript HTTP/1.1" 200 OK


Your max_length is set to 60, but your input_length is only 53. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=26)


transcript endpoint: 200
{'status': 'success', 'transcript': "All right, so here we are, in front of the elephants the cool thing about these guys is that they have really... really long trunks and that's cool (baaaaaaaaaaahhh!!) and that's pretty much all there is to say", 'word_count': 38}
INFO:     34.48.3.188:0 - "POST /summarize HTTP/1.1" 200 OK
summarize endpoint: 200
{'status': 'success', 'summary': "The elephants have really. really long trunks and that's cool (baaaaaaaaaaahhh!!) and that is pretty much all there is to say"}
INFO:     197.39.87.233:0 - "POST /transcript HTTP/1.1" 200 OK
INFO:     197.39.87.233:0 - "POST /summarize HTTP/1.1" 200 OK
INFO:     197.39.87.233:0 - "POST /transcript HTTP/1.1" 200 OK
INFO:     197.39.87.233:0 - "POST /summarize HTTP/1.1" 200 OK
INFO:     197.39.87.233:0 - "POST /transcript HTTP/1.1" 200 OK
INFO:     197.39.87.233:0 - "POST /summarize HTTP/1.1" 400 Bad Request
INFO:     197.39.87.233:0 - "POST /summarize HTTP/1.1" 400 Bad Request
INFO: 

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


INFO:     197.39.87.233:0 - "POST /summarize HTTP/1.1" 200 OK
INFO:     197.39.87.233:0 - "POST /summarize HTTP/1.1" 200 OK
